# CNN for Classifying EMG Muscle Contractions 

In [1]:
import os.path as op
import mne 
import os
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score,ConfusionMatrixDisplay, f1_score, classification_report  # Import necessary metrics
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import matplotlib
import pickle


from scipy.fft import fft, ifft,fftfreq
from scipy.signal import welch, find_peaks 

from tensorflow.keras.models import Sequential, Model  # Import Sequential model from TensorFlow Keras
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Input, Dropout  # Import necessary layers from TensorFlow Keras
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2, l1

matplotlib.use('QtAgg') 
mne.set_log_level("CRITICAL")

In [ ]:
# sanity check for columns to remove 
trigger_info = pd.read_csv("trigger_counts.csv")
print(trigger_info.loc[trigger_info["Trigger_number"] != 60, ["Subject", "Nap"]])

# Defining initial variables 

In [ ]:
to_keep=['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2','EOG1','EOG2','Corr', 'Zygo', 'Menton','Trigger']
eeg_ch= ['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2']
emg_ch= ['Corr', 'Zygo', 'Menton'] # Menton is chin EMG for sleep scoring
eog_ch= ['EOG1', 'EOG2']
trigger_ch= ['Trigger']
subject_exclude = ["NL01SS", "NL02IF",
                   "NL05WW01", "RL12JL03", "RL07BR02", "RL22AC05",
                   "RL16CM", "RL11JH","RL18CL", "RL21AC"]  # trials to exclude based on trigger count 


inter_trigger_length = 30
num_epochs = 60 # 60 epochs for each session 

raw_path= "/Users/zeynepozkaya/Desktop/Consciousness_Research/Python_Scripts/full_EEG_dataset"
current_index=0
inter_trigger_length=10
window = 50 
step = 1 

In [ ]:
# pre-processes data for each subject and returns df with session divided into 60 epochs w meta-data attached 
def pre_process_subjets(subject,block):
    global raw_path
    global frq

    subject_name = subject + block 
    file= op.join(raw_path,'{}.edf'.format(subject_name))
    raw =  mne.io.read_raw_edf(file,preload=True)

    if 'Fp1/F3' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'Fp1/F3':'Fp1' ,'Fp2/F4':'Fp2'})
        
    if '36' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'36':'Corr' ,'37':'Zygo','38':'Menton'})

    if 'E1' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'E1':'EOG1' ,'E2':'EOG2'})
    
    if 'Corru' in raw.info['ch_names']:
        raw.rename_channels({'Corru': 'Corr'})


    for ch in raw.info['ch_names']:
        if ch not in to_keep:
            raw.drop_channels([ch]) # only keeps channel that contains the trigger (where a stimulus was presented )

    raw.set_channel_types(mapping={'Corr':'emg','Zygo':'emg','Menton':'emg','Trigger':'stim','EOG1':'eog','EOG2':'eog'})

    raw= raw.resample(sfreq=250)

    filter_params_emg = {'lpass': 100,'hpass': 10,'notches': [50]}
    raw.filter(l_freq=filter_params_emg['hpass'],h_freq=filter_params_emg['lpass'],picks=emg_ch)
    filter_params_eeg_eog = {'lpass': 70,'hpass': 0.3,'notches': [50]}
    raw.filter(l_freq=filter_params_eeg_eog['hpass'],h_freq=filter_params_eeg_eog['lpass'],picks=eeg_ch+eog_ch)
    raw.notch_filter(filter_params_eeg_eog['notches'], method='fft', picks=emg_ch+eeg_ch+eog_ch)

    frq=raw.info['sfreq']

    # create dataframe based on events 
    events_all= mne.find_events(raw) # Get all the events (triggers) in your EEG data
    events= mne.pick_events(events_all,include=[201,202]) # Pick events of interest (where a stimulus was presented)

    df_triggers=pd.DataFrame(data=events,columns=['Time(Sample)','dunno','Trigger'])
    df_triggers.drop(columns=['dunno'],inplace=True)
    df_triggers['Time(s)']=df_triggers['Time(Sample)']/raw.info['sfreq']


    # incorporating meta_data 
    cols_inc = ["Subject","Nap_ID","Trigger", "Expected_Muscle","Nb_Corr","Nb_Zygo","Is_Correct"] # columns with relevant information from dataframe 
    trial_info = pd.read_csv("Trial_information_narcolepsy.csv", usecols=cols_inc) 
    expected_muscle = ["Corr", "Zygo"]


 
    epochs = mne.Epochs(raw, events, tmin=-0, tmax=9,baseline=None, detrend=0,
                    reject=None, preload=True, on_missing='warn')

   


    epochs.metadata = trial_info[(trial_info["Subject"] == subject) 
                                & (trial_info["Trigger"].isin([201.0, 202.0]))
                                & (trial_info["Nap_ID"] == int(block))]

    # adding metadata column for true activation
    # add another column for neither muscle being activated  
    true_activations = [] 
    for i in range(len(epochs.metadata)):
        true_ind = expected_muscle.index(epochs.metadata.iloc[i]["Expected_Muscle"]) 
        if (epochs.metadata.iloc[i]["Is_Correct"] == 0):
            if(epochs.metadata.iloc[i]["Nb_Corr"] < 3 and epochs.metadata.iloc[i]["Nb_Zygo"] < 3):
                true_activations.append("None")
            else:
                true_activations.append(expected_muscle[true_ind-1])
        else:
            true_activations.append(expected_muscle[true_ind])


    epochs.metadata["True_activation"] = true_activations
    return epochs, df_triggers 

In [ ]:
# gets features in a sliding window of size window samples with a step size of a certain number of samples 
def get_features(epoch):
     global frq
     global window 
     global step 

     # pad epoch to preserve sample number 
     pad_left  = window // 2
     pad_right = window - 1 - pad_left   
     
     epoch_padded = epoch
     epoch_padded = np.pad(epoch, (pad_left, pad_right), mode="edge")  

     # take sliding window 
     epoch_sw = np.lib.stride_tricks.sliding_window_view(epoch_padded,window)[::step]


     var = np.var(epoch_sw, axis=-1) # calculate variance over window 
     rms =  np.sqrt((1/window)*np.sum(epoch_sw**2, axis=-1)) # calculate rms over window  
     wl = np.sum(np.abs(np.diff(epoch_sw, axis=1)), axis=1) # calculate wl over window  
     mav = np.mean(np.abs(epoch_sw), axis=1)
     mavs = np.diff(mav)

     ''' 
     # frequency features 
    
     X = np.fft.rfft(epoch_sw, axis=1)
     PSD = (1/(frq*window)) * np.abs(X)**2
     cumulative = np.cumsum(PSD, axis=1)
     total_power = cumulative[:, -1]
     half_power = total_power / 2

     indices = (cumulative >= half_power[:, None]).argmax(axis=1)
     frequencies = np.fft.rfftfreq(window, 1/frq)

     fmd = frequencies[indices]
     '''

  
     return var, rms, wl, mavs


In [ ]:
def make_features_df(subject_epoch,subject,block):
    features = pd.DataFrame(
        index=range(num_epochs),
        columns=[
            "Subject",
            "Nap Number",
            "Triggers_Order_Nap", # epochs 
            "True_Muscle_Activated",
            "Num_Contractions_Zygo",
            "Num_Contractions_Corr",
            "WL_Zygo", # three features being used 
            "Var_Zygo",
            "RMS_Zygo", 
            "MAVS_Zygo",
            "WL_Corr",
            "Var_Corr",
            "RMS_Corr", 
            "MAVS_Corr",
            "Zygo", # processed EMG signal for epoch 
            "Corr"
        ]
        )   
    
    for t in range(len(subject_epoch)): 
        # extract epoch  
        epoch_zygo = np.squeeze(subject_epoch[t].get_data(picks=['Zygo']))
        epoch_corr = np.squeeze(subject_epoch[t].get_data(picks=['Corr']))

        epoch_var_zygo, epoch_rms_zygo, epoch_wl_zygo, epoch_mavs_zygo = get_features(epoch_zygo)
        epoch_var_corr, epoch_rms_corr, epoch_wl_corr, epoch_mavs_corr = get_features(epoch_corr)


        # fill dataframe 
        features.loc[t] = [
            subject, 
            int(block),
            t + 1,
            subject_epoch[t].metadata['True_activation'].iloc[0],
            subject_epoch.metadata.iloc[t]["Nb_Zygo"],
            subject_epoch.metadata.iloc[t]["Nb_Corr"],
            epoch_wl_zygo,
            epoch_var_zygo, 
            epoch_rms_zygo, 
            epoch_mavs_zygo,
            epoch_wl_corr,
            epoch_var_corr, 
            epoch_rms_corr, 
            epoch_mavs_corr,
            epoch_zygo, 
            epoch_corr
        ]
    
    return features 

Creating Dataframe

In [ ]:
#features_all = pd.concat(features_results_mat, ignore_index=True) # potentially save to excel and load in 
features_all.to_excel("training_features_19032026.xlsx", index=False)
#features_all.to_pickle("training_features_19032026.pkl")

In [28]:
# load in features
#features_all = pd.read_excel("training_features_11032026.xlsx")
features_all = pd.read_pickle("training_features_19032026.pkl")


# GUI

In [ ]:
subject_plt = "RL13MT"
nap_plt = "04"
subject_df = features_all.loc[(features_all["Subject"] == subject_plt) & (features_all["Nap Number"] == int(nap_plt))] # extract subj info

In [ ]:
#GUI
current_index=0
inter_trigger_length=10
window = 50
step = 1

def plot_figure(t):
    global frq 
    global subject_df


    # Muscle Activated: {epochs[t].metadata['True_activation'].iloc[0]}, Corr:{epochs[t].metadata['Nb_Corr'].iloc[0]}, Zygo:{epochs[t].metadata['Nb_Zygo'].iloc[0]}")
    
    zygo_contractions = np.array(subject_df["Num_Contractions_Zygo"])[t]
    corr_contractions = np.array(subject_df["Num_Contractions_Corr"])[t]

    epoch_zygo = np.array(subject_df["Zygo"].tolist())
    epoch_zygo_rms = np.array(subject_df["Zygo"].tolist())
    epoch_zygo_var = np.array(subject_df["Var_Zygo"].tolist())
    epoch_zygo_wl = np.array(subject_df["WL_Zygo"].tolist())

    epoch_corr = np.array(subject_df["Corr"].tolist())
    epoch_corr_rms = np.array(subject_df["RMS_Corr"].tolist())
    epoch_corr_var = np.array(subject_df["Var_Corr"].tolist())
    epoch_corr_wl = np.array(subject_df["WL_Corr"].tolist())


    ymax_features = np.ceil(np.max([np.max(epoch_corr_wl[t]), np.max(epoch_zygo_wl[t])]) / 100) * 100
    ymax_emg = np.ceil(np.max([np.max(epoch_corr[t]), np.max(epoch_zygo[t])]) / 100) * 100

    fig, ax1 = plt.subplots(2, 1) #, figsize=(10, 5))

    # create twin axes
    ax2_corr = ax1[0].twinx()
    ax2_zygo = ax1[1].twinx()

    fig.suptitle(f"Epoch {t + 1}")

    # Corr subplot
    ax1[0].plot(epoch_corr[t], color="blue", label="Corr")
    ''' 
    ax2_corr.plot(epoch_corr_rms[t], label="rms", color="orange")
    ax2_corr.plot(epoch_corr_var[t], label="var", color="red")
    ax2_corr.plot(epoch_corr_wl[t], label="wl", color="green")
    '''

    ax1[0].set_ylim(-200, 200)
    ax2_corr.set_ylim(0, ymax_features)

    print(zygo_contractions,corr_contractions)
    ax1[0].set_title(f"Zygo: {zygo_contractions}, Corr: {corr_contractions}")
    ax1[0].set_ylabel("Corr EMG [V]")
    ax2_corr.set_ylabel("Corr Features [V]")
    ax1[0].set_xlabel("Samples")

    # Zygo subplot
    ax1[1].plot(epoch_zygo[t], label="Zygo",color="black")
    '''
    ax2_zygo.plot(epoch_zygo_rms[t], label="rms", color="orange")
    ax2_zygo.plot(epoch_zygo_var[t], label="var", color="red")
    ax2_zygo.plot(epoch_zygo_wl[t], label="wl", color="green")
    '''

    ax1[1].set_ylim(-ymax_emg, ymax_emg)
    ax2_zygo.set_ylim(0, ymax_features)

    ax1[1].set_ylabel("Zygo EMG [V]")
    ax2_zygo.set_ylabel("Zygo Features [V]")
    ax1[1].set_xlabel("Samples")
    ax2_zygo.legend()  

 
 
    # Connect mouse click and key press events
    fig.canvas.mpl_connect('key_press_event', on_key)

    plt.tight_layout()
    plt.show()


# Keyboard press event handler
def on_key(event):
    global current_index, fig, features_all
    key = event.key
        
    if event.key == 'right':  # Move to next figure
        current_index = (current_index + 1) % len(subject_df)  # Loop to the start
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the next figure
    elif event.key == 'left':  # Move to previous figure
        current_index = (current_index - 1) % len(subject_df)  # Loop to the end
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the previous figure

    elif event.key == 'escape':  
        print("Quitting the plot!")
        plt.close(fig)  
        

# Plot the first figure
plot_figure(current_index)

# CNN 

Defining Model 

In [ ]:
del model

In [3]:
def CNN_model(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=1))  # Add another max pooling layer

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(num_classes, activation='softmax'))  # Add the output layer with softmax activation


    return model  # Return the compiled model

In [ ]:
def plot_accuracy(model_history,i):
    
    plt.plot(model_history.history['accuracy'], label='accuracy')
    plt.plot(model_history.history['val_accuracy'], label = 'val_accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.ylim([0.5, 1])
    plt.title(f"Fold {i}")
    plt.legend(loc='lower right')

Training with KFolds 

In [4]:
subjects = list(set(features_all["Subject"])) 

In [29]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       


indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [ ]:
print(input_shape)

In [6]:
# Create CNN model using the adjusted input shape and number of classes
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores=[]
epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model = CNN_model(input_shape, num_classes,feature_num)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])  

    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    
    model_history_kfold = model.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val)) #, callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model.evaluate(X_test,y_test)
    cvScores.append(scores[1] * 100)

    k += 1 
    

model_history = model.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test)) #, callbacks=[early_stop])

Fold: 1 ==================================================================


/Users/zeynepozkaya/anaconda3/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 28s 93ms/step - accuracy: 0.8207 - loss: 1.3467 - val_accuracy: 0.8633 - val_loss: 0.6049
Epoch 2/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 25s 88ms/step - accuracy: 0.8647 - loss: 0.5272 - val_accuracy: 0.8646 - val_loss: 0.6058
Epoch 3/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 26s 90ms/step - accuracy: 0.8970 - loss: 0.3715 - val_accuracy: 0.8733 - val_loss: 0.6488
Epoch 4/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 34s 118ms/step - accuracy: 0.9232 - loss: 0.2648 - val_accuracy: 0.8668 - val_loss: 0.7810
Epoch 5/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 37s 130ms/step - accuracy: 0.9454 - loss: 0.1892 - val_accuracy: 0.8720 - val_loss: 0.8688
Epoch 6/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 28s 95ms/step - accuracy: 0.9582 - loss: 0.1442 - val_accuracy: 0.8637 - val_loss: 0.9488
Epoch 7/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 23s 80ms/step - accuracy: 0.9718 - loss: 0.1006 - val_accuracy: 0.8589 - val_loss: 1.0449
Epoch 8/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 23s 81ms/step - accuracy: 0.9803 - loss: 0.0678 

In [8]:
model_history = model.fit(X_train_full, y_train_full, epochs=10, validation_data=(X_test, y_test)) #, callbacks=[early_stop])

Epoch 1/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 36s 100ms/step - accuracy: 0.9905 - loss: 0.0310 - val_accuracy: 0.9247 - val_loss: 0.6587
Epoch 2/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 33s 91ms/step - accuracy: 0.9908 - loss: 0.0370 - val_accuracy: 0.9167 - val_loss: 0.4293
Epoch 3/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 35s 97ms/step - accuracy: 0.9901 - loss: 0.0396 - val_accuracy: 0.9115 - val_loss: 0.6346
Epoch 4/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 29s 79ms/step - accuracy: 0.9920 - loss: 0.0304 - val_accuracy: 0.9240 - val_loss: 0.5349
Epoch 5/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 29s 80ms/step - accuracy: 0.9910 - loss: 0.0353 - val_accuracy: 0.9132 - val_loss: 0.5593
Epoch 6/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 30s 83ms/step - accuracy: 0.9907 - loss: 0.0316 - val_accuracy: 0.9160 - val_loss: 0.5829
Epoch 7/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 33s 91ms/step - accuracy: 0.9926 - loss: 0.0230 - val_accuracy: 0.9198 - val_loss: 0.5926
Epoch 8/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 29s 80ms/step - accuracy: 0.9936 - loss: 0.0237 -

In [14]:
# save model 
with open("M2_all_18042026.pkl", "wb") as f:
    pickle.dump(model, f)

In [25]:
with open("M2_all.pkl", "rb") as f:
    model = pickle.load(f)

In [9]:
# cross validation results 
avgScores = np.mean(cvScores)
stdScores = np.std(cvScores)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")

Average KFold Cross Validation Score: 95.20833373069763
Standard Deviation KFold Cross Validation Score: 1.3054816996484315


In [31]:
#mismatched_epochs1 = np.where(y[idx_test] != y_pred_test)
mismatched_epochs = np.where(y[idx_test] != y_pred_test)

print(mismatched_epochs)
np.shape(mismatched_epochs)

(array([  15,   17,   19,   33,   34,   42,   57,   64,  103,  114,  123,
        126,  145,  155,  215,  254,  259,  270,  275,  280,  283,  301,
        314,  358,  360,  373,  378,  383,  393,  402,  410,  426,  427,
        430,  439,  444,  457,  471,  478,  481,  488,  521,  525,  527,
        532,  557,  561,  563,  586,  592,  600,  614,  625,  651,  653,
        688,  694,  696,  723,  728,  734,  783,  791,  792,  804,  810,
        832,  838,  871,  877,  879,  885,  915,  934,  950,  959,  974,
        978, 1000, 1010, 1018, 1026, 1028, 1029, 1039, 1042, 1052, 1083,
       1084, 1118, 1127, 1158, 1196, 1215, 1217, 1219, 1233, 1266, 1269,
       1312, 1326, 1372, 1386, 1398, 1466, 1547, 1548, 1563, 1564, 1568,
       1574, 1589, 1603, 1613, 1623, 1624, 1628, 1631, 1660, 1695, 1713,
       1722, 1737, 1749, 1751, 1753, 1754, 1757, 1763, 1767, 1770, 1798,
       1808, 1813, 1822, 1826, 1829, 1830, 1831, 1833, 1836, 1837, 1846,
       1851, 1859, 1903, 1909, 1911, 1915, 1925, 1

(1, 220)

In [ ]:
print(np.array_equal(y_test, y[idx_test]))

In [30]:
# full training results (test data not seen during cross val)
y_pred_train = model.predict(X_train_full)  
y_pred_train = np.argmax(y_pred_train, axis=1)   

# Predict on test data
y_pred_test = model.predict(X_test)   
y_pred_test = np.argmax(y_pred_test, axis=1)   

# Calculate accuracy
accuracy_training = accuracy_score(y_train_full, y_pred_train)   
accuracy_test = accuracy_score(y_test, y_pred_test)  

# Calculate F1 score
f1_training = f1_score(y_train_full, y_pred_train, average='weighted')  
f1_test = f1_score(y_test, y_pred_test, average='weighted')  

# Print accuracy and F1 score
print("Training Accuracy :", accuracy_training)  
print("Test Accuracy :", accuracy_test)  
print("Training F1 Score :", f1_training)   
print("Test F1 Score :", f1_test)   

360/360 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step
Training Accuracy : 0.99921875
Test Accuracy : 0.9236111111111112
Training F1 Score : 0.9992120109222485
Test F1 Score : 0.9145928170556824


In [ ]:
# plotting model accuracy 
plt.plot(model_history.history['accuracy'], label='accuracy')
plt.plot(model_history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.7, 1])
plt.xlim([0,10])
plt.legend(loc='lower right')
plt.title("Model 1 Accuracy (all data)")
plt.show()

In [ ]:
# Confusion matrix 

cm = confusion_matrix(y_test, y_pred_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")

plt.show()

One Subject Test

In [ ]:
idx = features_all.index[features_all["Subject"] == subj_omit][0]

for t in range(idx,idx+60):
    subject_info = features_all.loc[t,["Subject"]].iloc[0]
    nap_info = features_all.loc[t,["Nap Number"]].iloc[0]
    print(subject_info,nap_info)


    sample_corr = features_all.loc[t,["Corr"]]
    sample_corr = sample_corr.to_numpy()
    sample_corr = np.array(sample_corr.tolist()).reshape(1, 2251,1)

    sample_zygo = features_all.loc[t,["Zygo"]]
    sample_zygo = sample_zygo.to_numpy()
    sample_zygo = np.array(sample_zygo.tolist()).reshape(1, 2251,1)

    prediction_corr = model.predict(sample_corr, verbose=0)
    prediction_zygo = model.predict(sample_zygo, verbose=0)

    corr_indices = np.argsort(prediction_corr[0].tolist())[-2:][::-1]
    
    cor_1 = corr_indices[0]
    cor_2 = corr_indices[1]
 
    print("Epoch: ",(t+1) % 60)
    print(f"Corr Contractions: {np.argmax(prediction_corr, axis=1)[0]}, Zygo Contractions: {np.argmax(prediction_zygo, axis=1)[0]}")
    print(f"Corr Contractions: {cor_1},{prediction_corr[0].tolist()[cor_1]}, {cor_2}, {prediction_corr[0].tolist()[cor_2]}")
    print("")